In [ ]:
from imblearn.over_sampling import SMOTE
import cv2
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, mean_squared_error, mean_absolute_error
from skimage.feature import hog
from skimage.color import rgb2gray
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

def extract_hog_features(image):
    """Extract HOG features from a single image."""
    # image = skimage.transform.resize(image.numpy(), (64, 128), anti_aliasing=True)
    gray_img = rgb2gray(image)
    features = hog(
        gray_img,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False
    )
    print('Shape: ', features.shape)
    sys.exit(0)
    return features

old = []
def load_images(image_dir, data):
    labels = []
    images = []
    gt_stats = []
    dirs = [image_dir + '/CS', image_dir + '/Healthy']
    for idx, image_dir in enumerate(dirs):
        for filename in sorted(os.listdir(image_dir)):
            if filename.endswith('.png'):
                print(filename)
                # Decode filename to extract sequence number, gender, and age
                # print(filename)
                seq_number = int(filename[:7]) 
                print(seq_number)
                # Find the corresponding row in the dataset
                row = data[data['pic_id'] == seq_number]

                if row.empty:
                    print(f"No matching row found for {filename}")
                    continue

                # Drop the Disease Classification column to use all other columns as label
                label_row = row.drop(columns=['pic_id']).iloc[0]
                # 
                gt_stats.append(label_row.values)

                img_path = os.path.join(image_dir, filename)
                image = cv2.imread(img_path)
                image = cv2.resize(image, (224, 224))

                labels.append(idx)
                images.append(image)
    
    
    return np.array(gt_stats), np.array(labels), np.array(images)/1.0  # Normalize images

# File paths to the dataset and image directories
# file_path = '/Users/srivatsavkannan/Datasets/C-Spine Xray/X-ray Atlas/results.xlsx'
file_path = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/results3.xlsx'
data = pd.read_excel(file_path, header=0).dropna()

train_image_dir = '/Users/srivatsavkannan/Datasets/Experiment/Train_Org_Cropped_ROS_RUS'
val_image_dir = '/Users/srivatsavkannan/Datasets/Experiment/Val_Org_Cropped_ROS_RUS'

# Load training and validation images and labels
X_train, y_train, images_train = load_images(train_image_dir, data)
X_val, y_val, images_val = load_images(val_image_dir, data)
hog_train = []
hog_val = []
for x in images_train:
    hog_train.append(extract_hog_features(x))
    
for x in images_val:
    hog_val.append(extract_hog_features(x))

smote = SMOTE(random_state=42)

hog_train = np.array(hog_train)
hog_val = np.array(hog_val)
# Apply ROS to the training and testing sets
# X_train, y_train = smote.fit_resample(X_train, y_train)
# X_val, y_val = smote.fit_resample(X_val, y_val)

print(X_train.shape)
print(X_val.shape)
print(images_train.shape)
print(hog_train.shape)

print(y_train.shape)
print(y_val.shape)
print(images_val.shape)
print(hog_val.shape)

In [ ]:
from imblearn.over_sampling import SMOTE
import cv2
import os
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras import layers, models

def extract_pca_features(image, pca):
    """Extract PCA features from a single image."""
    image_uint8 = (image * 255).astype(np.uint8)  # Convert back to uint8
    print(image_uint8.shape)
    image_gray = cv2.cvtColor(image_uint8, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    print(image_gray.shape)
    image_flat = image_uint8.flatten().reshape(1, -1)  # Flatten the image
    print(image_flat.shape)

    pca_features = pca.transform(image_flat)  # Apply PCA transformation
    return pca_features.flatten()


def load_images(image_dir, data):
    labels = []
    images = []
    gt_stats = []
    dirs = [image_dir + '/CS', image_dir + '/Healthy']
    for idx, image_dir in enumerate(dirs):
        for filename in sorted(os.listdir(image_dir)):
            if filename.endswith('.png'):
                seq_number = int(filename[:7])
                row = data[data['pic_id'] == seq_number]
                if row.empty:
                    continue
                label_row = row.drop(columns=['pic_id']).iloc[0]
                gt_stats.append(label_row.values)
                img_path = os.path.join(image_dir, filename)
                image = cv2.imread(img_path)
                image = cv2.resize(image, (224, 224))
                labels.append(idx)
                images.append(image)
    return np.array(gt_stats), np.array(labels), np.array(images) / 255.0  # Normalize images

# File paths to the dataset and image directories
file_path = '/Users/srivatsavkannan/Datasets/FinalCervicalDataset/results.xlsx'
data = pd.read_excel(file_path, header=0).dropna()

train_image_dir = '/Users/srivatsavkannan/Datasets/Experiment/Train_Org_Cropped_ROS_RUS'
val_image_dir = '/Users/srivatsavkannan/Datasets/Experiment/Val_Org_Cropped_ROS_RUS'

# Load training and validation images and labels
X_train, y_train, images_train = load_images(train_image_dir, data)
X_val, y_val, images_val = load_images(val_image_dir, data)

# Flatten images for PCA fitting
image_flattened_train = np.array([img.flatten() for img in images_train])
image_flattened_val = np.array([img.flatten() for img in images_val])

# Fit PCA on training images
pca = PCA(n_components=100)  # Adjust components as needed
pca.fit(image_flattened_train)

# Apply PCA transformation
pca_train = np.array([extract_pca_features(img, pca) for img in images_train])
pca_val = np.array([extract_pca_features(img, pca) for img in images_val])

smote = SMOTE(random_state=42)

# Apply SMOTE if needed
# X_train, y_train = smote.fit_resample(X_train, y_train)
# X_val, y_val = smote.fit_resample(X_val, y_val)

print(X_train.shape)
print(X_val.shape)
print(images_train.shape)
print(pca_train.shape)

print(y_train.shape)
print(y_val.shape)
print(images_val.shape)
print(pca_val.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_saint_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))

    # Feature Tokenization (Dense Layer for embedding features)
    x = layers.Dense(128, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    # Self-Attention Block
    for _ in range(2):  # 2 self-attention layers
        residual = x
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        x = layers.Add()([x, residual])  # Skip connection

    # Output Layer
    outputs = layers.Dense(1, activation='sigmoid')(x)  # Binary classification

    model = models.Model(inputs=inputs, outputs=outputs)
    return model
def build_efficientnet_hog_model(image_input_shape, hog_input_dim):
    """Multi-modal model with Tabular (SAINT), Image (EfficientNet), and HOG feature inputs."""
    # 🔹 Image Data Processing (EfficientNetB7)
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )
    base_model.trainable = False  # Freeze EfficientNet

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)  # Convert to 2D
    y = layers.Dense(512, activation='relu', name="CNN_Dense1")(y)
    y = layers.Dense(128, activation='relu', name="CNN_Dense2")(y)

    # 🔹 HOG Feature Processing
    hog_input = layers.Input(shape=(hog_input_dim,), name="hog_input")
    hog_x = layers.Dense(256, activation="relu", name="HOG_Dense1")(hog_input)
    hog_x = layers.Dense(128, activation="relu", name="HOG_Dense2")(hog_x)

    # 🔹 Combine Tabular, Image, and HOG Features
    combined = layers.Concatenate(name="Concatenated_Features")([y, hog_x])
    combined = layers.Dense(128, activation='relu', name="Combined_Dense")(combined)
    combined = layers.Dropout(0.3)(combined)

    # 🔹 Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(combined)

    # 🔹 Create the Model
    model = models.Model(inputs=[image_input, hog_input], outputs=outputs, name="SAINT_EfficientNet_HOG_Model")
    
    return model
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

def build_efficientnet_hog_model_l1(image_input_shape, hog_input_dim, l1_lambda=0.001):
    """Multi-modal model with Image (EfficientNet) and HOG feature inputs with L1 Regularization."""

    # 🔹 Image Data Processing (EfficientNetB7)
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.DenseNet161(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )
    base_model.trainable = False  # Freeze EfficientNet

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)  # Convert to 2D
    y = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l1(l1_lambda), name="CNN_Dense1")(y)
    y = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(l1_lambda), name="CNN_Dense2")(y)

    # 🔹 HOG Feature Processing
    hog_input = layers.Input(shape=(hog_input_dim,), name="hog_input")
    hog_x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l1(l1_lambda), name="HOG_Dense1")(hog_input)
    hog_x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l1(l1_lambda), name="HOG_Dense2")(hog_x)

    # 🔹 Combine Image and HOG Features
    combined = layers.Concatenate(name="Concatenated_Features")([y, hog_x])
    combined = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(l1_lambda), name="Combined_Dense")(combined)
    combined = layers.Dropout(0.3)(combined)

    # 🔹 Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(combined)

    # 🔹 Create the Model
    model = models.Model(inputs=[image_input, hog_input], outputs=outputs, name="EfficientNet_HOG_Model")

    return model

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

def build_efficientnet_pca_model(image_input_shape, pca_input_dim, l1_lambda=0.001):
    """Multi-modal model with Image (EfficientNet) and PCA feature inputs with L1 Regularization."""

    # 🔹 Image Data Processing (EfficientNetB7)
    image_input = layers.Input(shape=image_input_shape, name="image_input")
    base_model = tf.keras.applications.EfficientNetB7(
        include_top=False,
        weights='imagenet',
        input_shape=image_input_shape
    )
    base_model.trainable = False  # Freeze EfficientNet

    y = base_model(image_input, training=False)
    y = layers.GlobalAveragePooling2D()(y)  # Convert to 2D
    y = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l1(l1_lambda), name="CNN_Dense1")(y)
    y = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(l1_lambda), name="CNN_Dense2")(y)

    # 🔹 PCA Feature Processing (Replacing HOG)
    pca_input = layers.Input(shape=(pca_input_dim,), name="PCA_Input")
    pca_x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l1(l1_lambda), name="PCA_Dense1")(pca_input)
    pca_x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l1(l1_lambda), name="PCA_Dense2")(pca_x)

    # 🔹 Combine Image and PCA Features
    combined = layers.Concatenate(name="Concatenated_Features")([y, pca_x])
    combined = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l1(l1_lambda), name="Combined_Dense")(combined)
    combined = layers.Dropout(0.3)(combined)

    # 🔹 Output Layer (Binary Classification)
    outputs = layers.Dense(1, activation='sigmoid', name="Output_Layer")(combined)

    # 🔹 Create the Model
    model = models.Model(inputs=[image_input, pca_input], outputs=outputs, name="EfficientNet_PCA_Model")

    return model




In [ ]:
input_dim = X_train.shape[1]
image_input_dim = (224,224,3)
hog_input_dim = hog_train[0].shape[0]
print(type(hog_input_dim))
print(type(input_dim))
model = build_efficientnet_pca_model(image_input_dim, hog_input_dim)

X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
images_train = tf.convert_to_tensor(images_train, dtype=tf.float32)
hog_train = tf.convert_to_tensor(hog_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)

X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
images_val = tf.convert_to_tensor(images_val, dtype=tf.float32)
hog_val = tf.convert_to_tensor(hog_val, dtype=tf.float32)
y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model with class weights
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=100, restore_best_weights=True)
history = model.fit([images_train, hog_train], y_train,
                    validation_data=([images_val, hog_val], y_val),
                    epochs=20,
                    batch_size=32,
                    class_weight={0: 0.75, 1: 1.5},
                    callbacks=[early_stop],
                    verbose=1)

model.save("saint_77_trans_gen_balanced_hog_l1_cropped.keras")
model = tf.keras.models.load_model("saint_77_trans_gen_balanced_hog_l1_cropped.keras")


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_dim = X_train.shape[1]
image_input_dim = (224,224,3)
pca_input_dim = pca_train[0].shape[0]
print(type(pca_input_dim))
print(type(input_dim))
model = build_efficientnet_pca_model(image_input_dim, pca_input_dim)

X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
images_train = tf.convert_to_tensor(images_train, dtype=tf.float32)
pca_train = tf.convert_to_tensor(pca_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)

X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
images_val = tf.convert_to_tensor(images_val, dtype=tf.float32)
pca_val = tf.convert_to_tensor(pca_val, dtype=tf.float32)
y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model with class weights
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=100, restore_best_weights=True)
history = model.fit([images_train, pca_train], y_train,
                    validation_data=([images_val, pca_val], y_val),
                    epochs=20,
                    batch_size=32,
                    class_weight={0: 0.75, 1: 1.5},
                    callbacks=[early_stop],
                    verbose=1)

model.save("saint_77_trans_gen_balanced_pca_l1_cropped.keras")
model = tf.keras.models.load_model("saint_77_trans_gen_balanced_pca_l1_cropped.keras")


In [ ]:
from sklearn.metrics import confusion_matrix
from matplotlib import pyplot as plt
import seaborn as sns

train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
epochs = range(1, len(train_loss) + 1)

import json

# Save history
with open("training_history4.json", "w") as f:
    json.dump(history.history, f)

# Load history later
with open("training_history4.json", "r") as f:
    loaded_history = json.load(f)

# Access data
train_loss = loaded_history['loss']
val_loss = loaded_history['val_loss']
train_acc = loaded_history['accuracy']
val_acc = loaded_history['val_accuracy']


plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, 'b', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
#
# Plot training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, train_acc, 'b', label='Training accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()


# Predictions and metrics
# y_pred_train = (model.predict([X_train, images_train, hog_train]) > 0.5).astype(int)
y_pred_test = (model.predict([images_val, hog_val]) > 0.5).astype(int)
# y_pred_test_proba = model.predict([X_val, images_val, hog_val]).flatten()

# Compute metrics
# print("Training Classification Report:\n", classification_report(y_train, y_pred_train, digits=4))
class_names = ["CS", "Healthy"]
print("Testing Classification Report:\n", classification_report(y_val, y_pred_test, digits=4))
cm = confusion_matrix(y_val, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()
# Save the model


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib
import numpy as np

# Standardize the Tabular Data (SAINT Input)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Define Class Weights (Handle Imbalance)
class_weights = {0: 0.75, 1: 1.5}  # Adjust based on class distribution

# Train Logistic Regression Model with Class Weights
log_reg = LogisticRegression(class_weight=class_weights, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = log_reg.predict(X_train_scaled)
y_val_pred = log_reg.predict(X_val_scaled)

# Evaluation Metrics
train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

print("Classification Report (Validation):")
print(classification_report(y_val, y_val_pred, digits=4))

# Save Model & Scaler
joblib.dump(log_reg, "logistic_regression_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")


In [ ]:
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Standardize the Tabular Data (SAINT Input)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Define Class Weights (Handle Imbalance)
class_weights = {0: 0.75, 1: 1.5}  # Adjust based on class distribution

# Train XGBoost Model with Class Weights
xgb_classifier = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=class_weights[1] / class_weights[0],  # Adjust imbalance
    max_depth=6,
    learning_rate=0.05,
    n_estimators=500,
    eval_metric="logloss",
    use_label_encoder=False
)

xgb_classifier.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=True)

# Predictions
y_train_pred = xgb_classifier.predict(X_train_scaled)
y_val_pred = xgb_classifier.predict(X_val_scaled)

# Evaluation Metrics
train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

print("Classification Report (Validation):")
print(classification_report(y_val, y_val_pred, digits=4))

# Save Model & Scaler
joblib.dump(xgb_classifier, "xgboost_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Standardize the Tabular Data (SAINT Input)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Define Class Weights (Handle Imbalance)
class_weights = {0: 0.75, 1: 1.5}  # Adjust based on class distribution

# Train Random Forest Model with Class Weights
rf_classifier = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    class_weight=class_weights,  # Handle imbalance
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = rf_classifier.predict(X_train_scaled)
y_val_pred = rf_classifier.predict(X_val_scaled)

# Evaluation Metrics
train_acc = accuracy_score(y_train, y_train_pred)
val_acc = accuracy_score(y_val, y_val_pred)

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")

print("Classification Report (Validation):")
print(classification_report(y_val, y_val_pred, digits=4))

# Save Model & Scaler
joblib.dump(rf_classifier, "random_forest_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")
